<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z322_MARS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MARS — Multivariate Adaptive Regression Splines

## ¿Por qué MARS después de HAR?

HAR ajusta una **regresión lineal** con features de lags heterogéneos:
```
tn_t = c + β1·lag1 + β3·mean3 + β6·mean6 + β12·mean12
```

El supuesto implícito es que la relación es lineal y constante. MARS relaja eso:
- Encuentra **quiebres** automáticamente (splines adaptativos)
- Captura **no linealidades**: la tendencia puede importar distinto cuando las ventas son altas vs bajas
- Captura **interacciones**: `lag1 × mean12` si el pasado reciente interactúa con el nivel anual

Ejemplo de lo que MARS puede descubrir:
```
tn_t = c
      + β1 · max(0, lag1 - 100)      ← solo importa lag1 cuando supera 100
      + β2 · max(0, 50 - mean12)     ← efecto distinto cuando el nivel anual es bajo
      + β3 · max(0, lag1 - 200) · max(0, mean6 - 80)  ← interaccion
```

## Estructura del notebook

1. Test de Racha por producto (igual que en HAR)
2. Construccion de features HAR para MARS
3. Ajuste MARS por producto con `pyearth`
4. Prediccion recursiva a dos pasos
5. Submit a Kaggle

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle
# pyearth es la implementacion de MARS para Python
!uv pip install -q pyearth

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import runstest_1samp
from pyearth import Earth

import warnings
warnings.filterwarnings('ignore')

Por favor, cargar aqui SU semilla primigenia.

In [ ]:
PARAM = {
  'experimento': 'MARS-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  'alpha_runs': 0.05,
  # hiperparametros MARS
  'max_degree': 2,    # grado maximo de interacciones (1=sin interacciones, 2=pares)
  'max_terms': 10,    # maximo de terminos en el modelo final
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])
print(f"{tb_ventas.height} filas")

# 3  Test de Racha por producto

Mismo filtro que en HAR: series sin estructura van directo al promedio 12m, sin ajustar MARS.

In [ ]:
productos = tb_apredecir["product_id"].to_list()
runs_resultados = []

for producto in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    try:
        _, pvalue = runstest_1samp(serie, cutoff='median')
        tiene_estructura = bool(pvalue < PARAM['alpha_runs'])
    except Exception:
        pvalue = np.nan
        tiene_estructura = False

    promedio_12m = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    runs_resultados.append({
        'product_id': producto,
        'tiene_estructura': tiene_estructura,
        'promedio_12m': promedio_12m
    })

tb_runs = pl.DataFrame(runs_resultados)

n_estructura = tb_runs["tiene_estructura"].sum()
print(f"Con estructura (→ MARS)   : {n_estructura}")
print(f"Sin estructura (→ prom12m): {len(productos) - n_estructura}")

# 4  Features HAR para MARS

MARS recibe las mismas 4 features que HAR (`lag1`, `mean3`, `mean6`, `mean12`) pero en lugar de asumir linealidad, busca automáticamente:
- Quiebres en cada feature (hinge functions)
- Interacciones entre features si `max_degree=2`

Con solo 24 observaciones de entrenamiento (36 meses - 12 de arranque) y 4 features, MARS puede sobreajustar. Por eso limitamos `max_terms` y usamos `max_degree=2` con cuidado.

In [ ]:
def build_har_features(serie: np.ndarray):
    T = len(serie)
    rows_X, rows_y = [], []
    for t in range(12, T):
        lag1   = serie[t - 1]
        mean3  = serie[t-3 : t].mean()
        mean6  = serie[t-6 : t].mean()
        mean12 = serie[t-12: t].mean()
        rows_X.append([lag1, mean3, mean6, mean12])
        rows_y.append(serie[t])
    return np.array(rows_X), np.array(rows_y)


def mars_predict_next(modelo, serie: np.ndarray) -> float:
    t = len(serie)
    lag1   = serie[t - 1]
    mean3  = serie[t-3 : t].mean()
    mean6  = serie[t-6 : t].mean()
    mean12 = serie[t-12: t].mean()
    X_pred = np.array([[lag1, mean3, mean6, mean12]])
    return float(modelo.predict(X_pred)[0])

# 5  Ajuste MARS por producto

Usamos `pyearth.Earth` — la implementacion de MARS en Python.

Hiperparametros clave:
- `max_degree=1` → sin interacciones, solo quiebres lineales por feature (mas conservador, menos sobreajuste)
- `max_degree=2` → permite interacciones entre pares de features
- `max_terms` → limita la complejidad del modelo final

Con solo ~24 observaciones de entrenamiento, `max_degree=1` suele ser mas estable. Probá ambos.

In [ ]:
estructura_dict = dict(zip(
    tb_runs["product_id"].to_list(),
    tb_runs["tiene_estructura"].to_list()
))
promedio_dict = dict(zip(
    tb_runs["product_id"].to_list(),
    tb_runs["promedio_12m"].to_list()
))

resultados = []
modelos_ajustados = {}  # guardamos modelos para inspeccionar despues

for producto in productos:
    print(producto, end=' ')

    serie = (
        tb_ventas.filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    fallback = promedio_dict[producto]

    if not estructura_dict.get(producto, False):
        pred_2 = fallback
        metodo = 'promedio12m'
    else:
        try:
            X, y = build_har_features(serie)

            modelo = Earth(
                max_degree=PARAM['max_degree'],
                max_terms=PARAM['max_terms'],
                allow_missing=False
            )
            modelo.fit(X, y)
            modelos_ajustados[producto] = modelo

            # paso 1: predecir 202001
            pred_1 = max(mars_predict_next(modelo, serie), 0.0)

            # paso 2: predecir 202002
            serie_ext = np.append(serie, pred_1)
            pred_2 = max(mars_predict_next(modelo, serie_ext), 0.0)
            metodo = 'MARS'

        except Exception as e:
            print(f"  ERROR {producto}: {e}")
            pred_2 = fallback
            metodo = 'promedio12m_error'

    resultados.append({'product_id': producto, 'tn': pred_2, 'metodo': metodo})

print("\nlisto")

# 6  Inspeccion de modelos MARS

Una de las ventajas de MARS sobre modelos de caja negra es que podemos ver exactamente qué encontró el modelo en cada serie.

In [ ]:
# resumen de metodos usados
tb_resultado = pl.DataFrame(resultados)
display(tb_resultado.group_by("metodo").agg(pl.len().alias("n")))

In [ ]:
# ver el modelo de los primeros 3 productos con estructura
# summary() muestra los terminos que MARS encontro: quiebres e interacciones
for pid in list(modelos_ajustados.keys())[:3]:
    print(f"\n{'='*50}")
    print(f"product_id: {pid}")
    print(modelos_ajustados[pid].summary())

In [ ]:
# visualizacion: fitted vs real para algunos productos
pids_plot = list(modelos_ajustados.keys())[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, pid in enumerate(pids_plot):
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    X, y = build_har_features(serie)
    fitted = modelos_ajustados[pid].predict(X)

    axes[i].plot(range(len(y)), y, 'o-', color='steelblue', label='real', markersize=4, linewidth=1.5)
    axes[i].plot(range(len(fitted)), fitted, 's--', color='orange', label='MARS fitted', markersize=4, linewidth=1.5)
    axes[i].set_title(f"product_id {pid}", fontsize=9)
    axes[i].legend(fontsize=7)

fig.suptitle('MARS: fitted vs real (desde mes 13)', fontsize=11)
plt.tight_layout()
plt.show()

# 7  Submit a Kaggle

In [ ]:
tb_final = tb_resultado.select(["product_id", "tn"])

display(tb_final)
print(f"Total: {tb_final.height}  |  Nulls: {tb_final['tn'].is_null().sum()}")

In [ ]:
archivo = "MARS.csv"
mensaje = f"MARS degree={PARAM['max_degree']} terms={PARAM['max_terms']} + Racha + promedio12m"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 8  Que probar si el score no mejora

| Cambio | Donde | Por que |
|---|---|---|
| `max_degree=1` | PARAM | Menos sobreajuste con pocas obs |
| `max_terms=6` | PARAM | Modelo mas parsimonioso |
| `alpha_runs=0.10` | PARAM | Mas series van a MARS (menos van a promedio) |
| Agregar `lag2`, `lag6` individuales | `build_har_features` | Mas features para que MARS explore |
| Transformar con `log1p` | antes de ajustar | Estabiliza series con outliers grandes |